## Setup — paths and environment

In [1]:
import json
from pathlib import Path
from utils.utils import get_root_directory
from aieng.syn_data.synbench.display import (  # noqa: E402
    show_actions,
    show_pipeline_messages,
    show_session,
    show_task,
)

ROOT = get_root_directory()

DOMAIN_PATH = ROOT / "domains" / "mock_retail"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_retail" / "notebook_walkthrough"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)


Implementation root: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation
Domain: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_retail
Output: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_retail/notebook_walkthrough


---
## Step 1 — Load and inspect the domain

A **domain bundle** is the simulated business. `load_domain()` reads all required files and returns a `DomainBundle` object used everywhere else.

In [2]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

print("Domain name:", domain.manifest.name)
print("\nTools:")
for t in domain.tools:
    print(f"  - {t.name} ({t.tool_type.value}, fsm_tag={t.fsm_tag})")
print("\nTask types (FSM):", list(domain.state_machine.get("task_types", {}).keys()))
print("\nSeed task IDs:", [t.id for t in domain.seed_tasks])
print("\nOrders in DB:", list(domain.db["orders"].keys()))

Domain name: mock_retail

Tools:
  - find_user_id (read, fsm_tag=lookup)
  - get_order (read, fsm_tag=lookup)
  - list_orders (read, fsm_tag=lookup)
  - cancel_order (write, fsm_tag=mutate)
  - update_shipping (write, fsm_tag=mutate)

Task types (FSM): ['inquiry', 'cancel', 'update_address', 'refuse_cancel']

Seed task IDs: ['seed_inquiry', 'seed_cancel', 'seed_refuse_cancel', 'seed_update_address']

Orders in DB: ['ord_1001', 'ord_1002', 'ord_2001']


In [3]:
# Peek at policy and one order
print("=== Policy (first 400 chars) ===")
print(domain.policy[:400], "...\n")

print("=== Sample order ord_1001 ===")
print(json.dumps(domain.db["orders"]["ord_1001"], indent=2))

=== Policy (first 400 chars) ===
# Mock Retail Customer Service Policy

You are a customer service agent for Mock Retail.

## Rules

1. Customers do not know their `user_id`. Always ask for the customer's full
   name first, then call `find_user_id` with that name to retrieve their
   `user_id`. Never invent, guess, or fabricate a `user_id`.
2. If `find_user_id` fails (user not found), tell the customer no account
   matches that ...

=== Sample order ord_1001 ===
{
  "order_id": "ord_1001",
  "user_id": "user_alice",
  "status": "pending",
  "items": [
    "widget-a"
  ],
  "shipping_address": "123 Main St, Boston, MA"
}


---
## Step 2 - Load and re-verify previously generated tasks

In [3]:
# Re-verify all the generated tasks in the saved file.
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


task_path = OUT_DIR / "tasks.json"
with open(task_path) as f:
    saved = json.load(f)

filtered_tasks = []
for item in saved["tasks"]:
    task = Task.model_validate(item)
    filtered_tasks.append(task)
    vr = verify_draft(domain, task)
    status = "OK" if vr.verification_report.passed else "FAIL"
    print(f"  [{status}] {task.id}")

# Do you see any FAILED tasks?
# If yes, import filter_verified from synbench.verification.pipeline and call it
# on (domain, filtered_tasks) to drop the failures before evaluating.

  [OK] gen_dc547ce1_0
  [OK] gen_47075073_1
  [OK] gen_66bde9b2_2


---
## Step 3 - LLM layer (provider-agnostic agent core)

Agents and generation share `synbench.llm`. **`get_client()`** returns an OpenAI-compatible `ChatClient` (default model: `gemini-2.0-flash`).

Make sure to run the first notebook to check your API key works.


In [ ]:
from aieng.syn_data.synbench.llm import get_client, get_model
from aieng.syn_data.synbench.llm.tools import tool_specs_to_openai


client = get_client()
print("Model:", get_model())
print("Client type:", type(client).__name__)
print("\nOpenAI-style tool schema (cancel_order):")
print(json.dumps(tool_specs_to_openai(domain.tools)[3], indent=2))

---
## Step 4 — Multi-agent pipeline

`AgentPipeline` uses the same dialogue helper as `SingleToolAgent`, plus planner/critic.

Per dialogue turn:

| Role | Purpose | Calls tools? |
|------|---------|--------------|
| **user_sim** | Simulates the customer (`user_name` + `personality_style`) | No |
| **planner** | Produces a short plan | No |
| **executor** | Runs `ToolCallingLoop` | Yes |
| **critic** | Reviews plan + trace + reply; may trigger one retry | No |

- Turn 0 uses `initial_message`; later turns call user_sim until `[[DONE]]` or `max_dialogue_turns`
- Only the **executor** modifies the database
- The agent under test does not see clean `instructions`


In [5]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline


print("Default role order:", DEFAULT_ROLES)

# max_turns = tool rounds; max_dialogue_turns = user ↔ pipeline rounds
pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=3,
    client=client,
)

task0 = filtered_tasks[0]
print(f"Task {task0.id}: user={task0.user_scenario.user_name!r}, style={task0.user_scenario.personality_style!r}")
pipe_score = pipeline.run_and_score_task(task0)
print("\n=== Pipeline score ===")
print(f"  reward:     {pipe_score.reward}")
print(f"  db_reward:  {pipe_score.db_reward}")
print(f"  comm:       {pipe_score.communicate_reward}")

Default role order: ['user_sim', 'planner', 'executor', 'critic']
Task gen_dc547ce1_0: user='Alice Chen', style='rule_breaker'

=== Pipeline score ===
  reward:     1.0
  db_reward:  1.0
  comm:       1.0


### [Optional] — Inspect a full pipeline dialogue session


In [6]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline


# Fresh pipeline so we can inspect role_trace / messages after dialogue
pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=3,
    client=client,
)
pipeline_session = pipeline.run_task(filtered_tasks[0])
print("Role trace:", pipeline_session.role_trace)

Role trace: ['planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic']


In [7]:
# Inspect task (user_name + personality_style), session, messages, actions
show_task(pipeline_session.task, title="Task")
show_session(pipeline_session, title="Session (after dialogue)", show_chat=False)
show_pipeline_messages(pipeline_session, title="Pipeline messages")
show_actions(pipeline_session.agent_actions, title="Agent actions")

### Task
**id:** `gen_dc547ce1_0`  
**type:** `inquiry`  
**description:** Look up order details for the user

#### User scenario
- **user name:** Alice Chen
- **personality style:** rule_breaker
- **instructions:** Inquire about the status of order ord_1001. Do not volunteer the order ID immediately; force the agent to ask for it.
- **initial message:** Listen, I need to know what's going on with my delivery. It's been way too long and I bet you guys lost it. Can you check my account?

#### Evaluation criteria
- **reward basis:** DB, COMMUNICATE
- **communicate info:** []
- **oracle actions:**
  - `get_order` `{"order_id": "ord_1001"}`

### Session (after dialogue)

| Field | Value |
| --- | --- |
| **domain** | `mock_retail` (`/Users/fatemehtavakoli/Desktop/synthetic data generation/agentic/domains/mock_retail`) |
| **task** | `gen_dc547ce1_0` · `inquiry` |
| **env** | ready |
| **messages** | 11 |
| **agent_actions** | 2 |
| **agent_messages** | 3 |
| **role_trace** | ['planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic'] |
| **plan** | — |

### Agent actions
1. `find_user_id` — `{"name": "Alice Chen"}`
2. `list_orders` — `{"user_id": "user_alice"}`

### Agent replies

> I would be happy to help you check the status of your order. To get started, could you please provide your full name?

> I found two orders associated with your account, Alice:
> 
> 1. **Order ord_1001**: Contains one widget-a. It is currently **pending** and scheduled to be shipped to 123 Main St, Boston, MA.
> 2. **Order ord_1002**: Contains one widget-b. It has been **shipped** to 456 Oak Ave, Boston, MA.
> 
> Could you please confirm which order you are inquiring about?

> I understand your frustration with the wait, Alice. While I cannot manually expedite the processing of an order, I can confirm that your order **ord_1001** (widget-a) is currently in **pending** status for shipment to **123 Main St, Boston, MA**.
> 
> Since it is still pending, it is currently being prepared for shipment. Would you like me to keep this order as is, or did you have any other concerns regarding the shipping address or the order itself?

### Pipeline messages

**role_trace:** `planner → executor → critic → user_sim → planner → executor → critic → user_sim → planner → executor → critic`

**[0] system**
    You are the customer service agent for this domain.
    task_id: gen_dc547ce1_0

    ## Policy
    # Mock Retail Customer Service Policy

    You are a customer service agent for Mock Retail.

    ## Rules

    1. Customers do not know their `user_id`. Always ask for the customer's full
       name first, then call `find_user_id` with that name to retrieve their
       `user_id`. Never invent, guess, or fabricate a `user_id`.
    2. If `find_user_id` fails (user not found), tell the customer no account
       matches that name and ask them to confirm they typed their full name
       correctly. Do not invent a `user_id` or continue with order tools until
       a match is found.
    3. Use the retrieved `user_id` with `list_orders` (and when verifying ownership)
       before answering account-level questions or modifying orders.
    4. After looking up an order, confirm the order details with the customer —
       including items and shipping address — before canceling, updating shipping,
       or treating the lookup as complete.
    5. Always look up an order before canceling or updating shipping.
    6. Only cancel orders with status `pending`. Shipped or delivered orders cannot
       be canceled.
    7. If a customer asks to cancel a non-pending order, politely refuse and explain
       why.
    8. When updating shipping, confirm the new address with the customer in your
       message.
    9. Never modify orders that do not belong to the requesting user.

    ## Communication

    - Be concise and professional.
    - When refusing a cancellation, state that the order cannot be canceled due to its status.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

**[1] user**
    Listen, I need to know what's going on with my delivery. It's been way too long and I bet you guys lost it. Can you check my account?

**[2] assistant (executor)**
    I would be happy to help you check the status of your order. To get started, could you please provide your full name?

**[3] user**
    My name is Alice Chen. Look, I’ve been waiting forever, so don’t give me the runaround. Just look me up in your system and tell me why my package isn't at my door yet.

**[4] assistant (executor)**
    tool_call `find_user_id`
    {
      "name": "Alice Chen"
    }

**[5] tool**
    {
      "user_id": "user_alice",
      "name": "Alice Chen",
      "email": "alice@example.com"
    }

**[6] assistant (executor)**
    tool_call `list_orders`
    {
      "user_id": "user_alice"
    }

**[7] tool**
    [
      {
        "order_id": "ord_1001",
        "user_id": "user_alice",
        "status": "pending",
        "items": [
          "widget-a"
        ],
        "shipping_address": "123 Main St, Boston, MA"
      },
      {
        "order_id": "ord_1002",
        "user_id": "user_alice",
        "status": "shipped",
        "items": [
          "widget-b"
        ],
        "shipping_address": "456 Oak Ave, Boston, MA"
      }
    ]

**[8] assistant (executor)**
    I found two orders associated with your account, Alice:

    1. **Order ord_1001**: Contains one widget-a. It is currently **pending** and scheduled to be shipped to 123 Main St, Boston, MA.
    2. **Order ord_1002**: Contains one widget-b. It has been **shipped** to 456 Oak Ave, Boston, MA.

    Could you please confirm which order you are inquiring about?

**[9] user**
    It’s the one for ord_1001, obviously. I don't care why it's "pending"—that's just a fancy word for lazy. I need you to expedite this right now and get it sent out today. I’m not waiting any longer for this. Can you just push it through for me?

**[10] assistant (executor)**
    I understand your frustration with the wait, Alice. While I cannot manually expedite the processing of an order, I can confirm that your order **ord_1001** (widget-a) is currently in **pending** status for shipment to **123 Main St, Boston, MA**.

    Since it is still pending, it is currently being prepared for shipment. Would you like me to keep this order as is, or did you have any other concerns regarding the shipping address or the order itself?


### Agent actions
1. `find_user_id` — `{"name": "Alice Chen"}`
2. `list_orders` — `{"user_id": "user_alice"}`

---
## Step 6 — Batch metrics (pass@1)

`MetricsCollector` aggregates scores across tasks.

In [8]:
from aieng.syn_data.synbench.evaluation.metrics import MetricsCollector


# Evaluate generated tasks with multi-agent pipeline
metrics = MetricsCollector()
for task in filtered_tasks:
    metrics.add(task.id, pipeline.run_and_score_task(task))

summary = metrics.summary()
print(json.dumps(summary, indent=2))

{
  "n_tasks": 3,
  "pass_at_1": 1.0,
  "mean_db_reward": 1.0,
  "mean_communicate_reward": 1.0,
  "mean_partial_action_match": 0.6666666666666666,
  "runs": [
    {
      "task_id": "gen_dc547ce1_0",
      "reward": 1.0,
      "db_reward": 1.0,
      "communicate_reward": 1.0,
      "partial_action_match": 0.0
    },
    {
      "task_id": "gen_47075073_1",
      "reward": 1.0,
      "db_reward": 1.0,
      "communicate_reward": 1.0,
      "partial_action_match": 1.0
    },
    {
      "task_id": "gen_66bde9b2_2",
      "reward": 1.0,
      "db_reward": 1.0,
      "communicate_reward": 1.0,
      "partial_action_match": 1.0
    }
  ]
}
